# 🧹 Module 3 — Class 1: Clean the Customer Data

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 The story

- Monday morning. You work at a telecom in Tashkent.
- **Boss:** *"Tell me who will leave next month."*
- **Data team:** *"Here's `customers.csv`. We merged 3 systems. It's a mess. Fix it."*

**Today we clean. Tomorrow we train.**

## 💡 Why does this matter?

- Bad data → wrong predictions → angry boss
- Real data is **always** messy
- Cleaning = 80% of a data scientist's work

## 🎯 Today's plan

1. **Look** at the data
2. **Fix** the money column
3. **Fill** empty values
4. **Same word** for same thing
5. **Remove** duplicate customers
6. **Fix** weird numbers
7. **Save** the clean file
8. **Bonus:** apply to real Telco data

## 🤖 How to run

Click a code cell → press **Shift + Enter**.

👉 Each code cell will tell you **what to look at** AND **what it means**. Read the print messages.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
print('✅ Tools ready! pandas, numpy, matplotlib are now imported.')

---
## Step 1: Get the file 📥

We make a fake (but realistic) dirty file. Real customer data is private — we can't share it.

**Just run the next cell.** Don't worry about how it works.

In [ ]:
# 🔒 Don't read this. Just run it. It makes a dirty customer file.
def make_dirty_customers():
    rng = np.random.default_rng(42)
    n = 3000
    names  = ['Ali','Zara','Jasur','Madina','Otabek','Aisha','Karim','Nargiza',
              'Bobur','Dilnoza','Sardor','Malika','Timur','Shaxnoza','Rustam','Feruza']
    cities = ['Tashkent','Samarkand','Bukhara','Andijan','Namangan','Fergana']
    plans  = ['basic','pro','premium']
    nets   = ['Fiber','DSL','No internet service']
    df = pd.DataFrame({
        'customer_id':       [f'CU{i:05d}' for i in range(1, n+1)],
        'name':              rng.choice(names, n),
        'city':              rng.choice(cities, n),
        'age':               rng.integers(18, 75, n).astype(float),
        'tenure_months':     rng.integers(1, 73, n).astype(float),
        'monthly_charges':   rng.normal(80, 25, n).round(2),
        'plan':              rng.choice(plans, n),
        'internet_service':  rng.choice(nets, n, p=[0.45,0.35,0.20]),
        'has_phone_service': rng.choice(['Yes','No phone service'], n, p=[0.9,0.1]),
        'churn':             rng.choice(['Yes','No'], n, p=[0.27,0.73])
    })
    df['total_charges'] = (df['tenure_months'] * df['monthly_charges']).round(2)
    df.loc[rng.choice(n, 80, replace=False), 'tenure_months']  = np.nan
    df.loc[rng.choice(n, 60, replace=False), 'monthly_charges']= np.nan
    df.loc[rng.choice(n, 40, replace=False), 'age']            = np.nan
    df['total_charges'] = df['total_charges'].astype(str)
    df.loc[rng.choice(n, 25, replace=False), 'total_charges'] = ' '
    ti = df.index[df['city']=='Tashkent'].tolist(); rng.shuffle(ti)
    df.loc[ti[:80],'city']='tashkent'; df.loc[ti[80:140],'city']='TASHKENT'
    df.loc[ti[140:200],'city']=' Tashkent '
    pi = df.index[df['plan']=='pro'].tolist(); rng.shuffle(pi)
    df.loc[pi[:50],'plan']='PRO'; df.loc[pi[50:80],'plan']='Pro '
    yi = df.index[df['churn']=='Yes'].tolist(); rng.shuffle(yi)
    df.loc[yi[:30],'churn']='yes'; df.loc[yi[30:60],'churn']='YES'
    df.loc[yi[60:90],'churn']=' Yes '
    df.loc[100,'tenure_months']=-3; df.loc[200,'tenure_months']=999
    df.loc[300,'monthly_charges']=99999; df.loc[400,'age']=200
    df = pd.concat([df, df.iloc[[5,50,100,500,1000,2000]]], ignore_index=True)
    return df

df = make_dirty_customers()
print(f'📄 Got file from data team!')
print(f'   Total rows: {len(df)} customers')
print(f'   Total columns: {df.shape[1]} fields per customer')
print()
print('👉 Next: look at the first 5 rows to see what we got.')

In [ ]:
# Show first 5 rows
print('👀 First 5 rows of the file:')
print('   Each row = one customer.')
print()
df.head()

👀 **It looks fine, right?** Trust me — there are problems hiding inside.

Important columns:
- `tenure_months` — how long they've been our customer
- `monthly_charges` — what they pay each month
- `churn` — did they leave? (Yes / No) ← **what we want to predict**

---
## Step 2: Look 👀

> **Always look before you change.** A doctor checks before they cut.

Three commands tell us a lot:

In [ ]:
# Show column types and how many values are NOT empty
print('👀 Looking at column types and non-empty counts:')
print()
df.info()
print()
print('━' * 50)
print('🚨 PROBLEMS FOUND:')
print('  ❌ total_charges is "object" (text) → should be a number!')
print('  ❌ tenure_months has empty values (less than total)')
print('  ❌ monthly_charges has empty values')
print('  ❌ age has empty values')
print('━' * 50)
print()
print('💡 Types you will see:')
print('   int64   = whole number (5)')
print('   float64 = number with a dot (19.95)')
print('   object  = text')

In [ ]:
# Show numbers about NUMBER columns
print('📊 Numbers about NUMBER columns')
print('   Pay attention to the min and max rows!')
print()
df.describe()

In [ ]:
# Now explain what we just saw
print('━' * 50)
print('🚨 WEIRD NUMBERS FOUND in the table above:')
print()
print('  ❌ tenure_months min = -3   (negative months? IMPOSSIBLE!)')
print('  ❌ tenure_months max = 999  (999 months = 83 years? IMPOSSIBLE!)')
print('  ❌ monthly_charges max = 99,999  (typo? VIP customer?)')
print('  ❌ age max = 200  (200 years old? IMPOSSIBLE!)')
print()
print('💡 LESSON: Always check min and max. Bad data hides there!')
print('━' * 50)

In [ ]:
# Count empty values per column
print('🕳️ Empty values per column:')
print()
missing = df.isnull().sum()
print(missing[missing > 0])
print()
print('━' * 50)
print(f'👉 Total empty values: {missing.sum()} across 3 columns')
print('   We will fill these in Step 4.')
print('━' * 50)

In [ ]:
# Look at city column
print('🏙️ Cities in our data:')
print()
print(df['city'].value_counts())
print()
print('━' * 50)
print('🚨 PROBLEM: 4 spellings of Tashkent!')
print('   Tashkent     ← original')
print('   tashkent     ← from a different signup form')
print('   TASHKENT     ← phone keyboard auto-caps')
print("    Tashkent     ← extra spaces from Excel")
print()
print('💡 The model thinks they are 4 different cities! Wrong!')
print('   We will fix this in Step 5.')
print('━' * 50)

### 🤔 Stop and think

Before we start fixing, can you answer these?

1. How many rows? How many columns?
2. Which column is `object` but should be a number?
3. How many empty values total?
4. What is the smallest tenure? Why is that a problem?

---
## Step 3: Fix the money column 💰

> **Why?** Math doesn't work on text. `"5" + "3" = "53"` (wrong!).

Find the bad rows first.

In [ ]:
# Find rows that aren't real numbers
bad = pd.to_numeric(df['total_charges'], errors='coerce').isna()

print(f'🔍 Found {bad.sum()} bad rows in total_charges')
print()
print('Sample of bad rows:')
print(df.loc[bad, ['customer_id', 'total_charges']].head())
print()
print('━' * 50)
print("💡 AHA! The bad value is ' ' (a blank space).")
print('   That is why pandas read everything as text.')
print('   Even ONE blank value makes the WHOLE column become text!')
print('━' * 50)

In [ ]:
# Step 1: convert text to number (bad becomes NaN)
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# Step 2: fill NaN with the median
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

print('✅ FIXED total_charges!')
print()
print(f'   Type now: {df["total_charges"].dtype}  (was: object)')
print(f'   Empty values now: {df["total_charges"].isna().sum()}  (was: 25)')
print(f'   Mean: {df["total_charges"].mean():.2f}  ← we can do math now!')
print()
print('💡 errors="coerce" turned blank " " into NaN.')
print('   Then fillna(median) replaced NaN with the middle value.')

---
## Step 4: Fill empty values 🕳️

> **Why?** Models crash on empty values.

Three ways to fill:

| Way | Best for |
| --- | --- |
| **Mean** (average) | Numbers, no big outliers |
| **Median** (middle) | Numbers, **safer** with outliers |
| **Mode** (most common) | Text columns |

In [ ]:
# Show empty values per column
missing = df.isnull().sum()
print('🕳️ Empty values per column (before fix):')
print()
print(missing[missing > 0])
print()
print(f'👉 Total: {missing.sum()} empty values to fill')

In [ ]:
# Fill 3 columns with their median
print('🔧 Filling empty values with the median...')
print()
for col in ['tenure_months', 'monthly_charges', 'age']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'   {col:18s} → filled with median = {median_val:.1f}')

print()
print(f'✅ Done! Empty values now: {df.isnull().sum().sum()}')
print()
print('💡 Why median, not mean?')
print('   We have a 99,999 outlier in monthly_charges.')
print('   The mean would be too high. Median ignores outliers.')

### 📝 YOUR TURN — make a 'was empty' column

Sometimes 'this was empty' is useful info!

Example: a customer who didn't fill in `monthly_charges` → maybe they don't trust the form → maybe they will leave!

**Trick:** add a 'was missing' column **BEFORE** filling.

**WARNING:** if you do it AFTER fillna, all values are False — too late!

```python
df_raw = make_dirty_customers()
df_raw['tenure_was_missing'] = df_raw['tenure_months'].isna().astype(int)
# .astype(int) changes True/False to 1/0
```

In [ ]:
# YOUR CODE here


---
## Step 5: Same word, same meaning 🧽

> **Why?** `'Yes'` and `'yes'` and `' Yes '` all mean Yes. Model thinks 3 categories.

**The 2-step pattern:**
```python
df['col'].str.strip().str.lower()
```
- `.str.strip()` removes spaces at start and end
- `.str.lower()` makes lowercase

In [ ]:
# Show city values BEFORE cleaning
print('🏙️ city BEFORE cleaning:')
print()
print(df['city'].value_counts())
print()
print(f'👉 We have {df["city"].nunique()} different city values.')
print('   But there are only 6 real cities!')

In [ ]:
# Clean 3 text columns: strip spaces, then lowercase
for col in ['city', 'plan', 'churn']:
    df[col] = df[col].str.strip().str.lower()

print('🏙️ city AFTER cleaning:')
print()
print(df['city'].value_counts())
print()
print('━' * 50)
print(f'✅ Fixed! Now {df["city"].nunique()} cities (was {9}).')
print('   All 4 spellings of Tashkent → one: "tashkent"')
print('━' * 50)

In [ ]:
# Check the other cleaned columns too
print('📋 plan values now:')
print(df['plan'].value_counts())
print()
print('📋 churn values now:')
print(df['churn'].value_counts())
print()
print('✅ Both columns now have only the correct values.')

In [ ]:
# Fix verbose value: 'No phone service' really means 'No'
before = df['has_phone_service'].unique().tolist()
df['has_phone_service'] = df['has_phone_service'].replace('No phone service', 'No')
after = df['has_phone_service'].unique().tolist()

print('📞 has_phone_service:')
print(f'   Before: {before}')
print(f'   After:  {after}')
print()
print("✅ Collapsed 2 categories to 2 (one is real 'No' now)")
print()
print("💡 Why? 'No phone service' and 'No' both mean the customer doesn't have phone.")

---
## Step 6: Remove duplicate customers 👯

> **Why?** Same customer logged twice = wrong metrics. Model 'memorizes' that customer.

Use `customer_id` as the key (not the whole row).

In [ ]:
# Count duplicates BEFORE
dup_count = df.duplicated(subset=['customer_id']).sum()
rows_before = len(df)

print('🔍 Checking for duplicate customers...')
print(f'   Rows before: {rows_before}')
print(f'   Duplicate customer_ids: {dup_count}')
print()

# Drop duplicates. Keep first.
df = df.drop_duplicates(subset=['customer_id']).reset_index(drop=True)

print(f'✅ Removed {dup_count} duplicates')
print(f'   Rows now: {len(df)}')
print()
print('💡 We dedupe by customer_id, NOT by full row.')
print('   Same customer with a typo in name = still the same customer!')

---
## Step 7: Fix weird numbers 🚨

Three kinds of outliers:

| Kind | Example | What to do |
| --- | --- | --- |
| **Mistake** | tenure = -3 | Remove or fix |
| **Real but big** | VIP customer | Keep, maybe cap |
| **Fraud** | wants to be detected | **Never** remove |

🚨 **Always think first!**

In [ ]:
# Step 1: Find IMPOSSIBLE values
bad_tenure = (df['tenure_months'] < 0) | (df['tenure_months'] > 120)
bad_age    = (df['age'] < 0) | (df['age'] > 120)

print('🔍 Looking for impossible values...')
print()
print(f'   Bad tenure values: {df.loc[bad_tenure, "tenure_months"].tolist()}')
print(f'   Bad age values   : {df.loc[bad_age, "age"].tolist()}')
print()
print('💡 Realistic limits we used:')
print('   - tenure: 0 to 120 months')
print('   - age   : 0 to 120 years')

In [ ]:
# Replace impossible with NaN, then fill with median
df.loc[bad_tenure, 'tenure_months'] = np.nan
df.loc[bad_age,    'age']           = np.nan
df['tenure_months'] = df['tenure_months'].fillna(df['tenure_months'].median())
df['age']           = df['age'].fillna(df['age'].median())

print('✅ Fixed impossible values!')
print(f'   tenure now: min={df["tenure_months"].min():.0f}, max={df["tenure_months"].max():.0f}')
print(f'   age now   : min={df["age"].min():.0f}, max={df["age"].max():.0f}')
print()
print('💡 We replaced impossible values with NaN, then filled with median.')
print('   Same trick as Step 4!')

**Now find statistical outliers (IQR rule):**

```
Q1 = 25% point, Q3 = 75% point, IQR = Q3 - Q1
low  = Q1 - 1.5 × IQR
high = Q3 + 1.5 × IQR
outside [low, high] = outlier
```

In [ ]:
# Find statistical outliers in monthly_charges (IQR rule)
Q1 = df['monthly_charges'].quantile(0.25)
Q3 = df['monthly_charges'].quantile(0.75)
IQR = Q3 - Q1
low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = (df['monthly_charges'] < low) | (df['monthly_charges'] > high)

print('🔍 IQR outlier detection on monthly_charges:')
print(f'   Q1 (25%) = {Q1:.2f}')
print(f'   Q3 (75%) = {Q3:.2f}')
print(f'   IQR      = {IQR:.2f}')
print(f'   Limits   = [{low:.1f}, {high:.1f}]')
print()
print(f'🚨 Outliers found: {outliers.sum()}')
print(f'   Biggest value : {df["monthly_charges"].max():.0f}  ← that\'s the 99,999 from data entry!')

In [ ]:
# Cap (winsorize) — keep customer, prevent extreme value
max_before = df['monthly_charges'].max()
df['monthly_charges'] = df['monthly_charges'].clip(lower=low, upper=high)
max_after  = df['monthly_charges'].max()

print('✅ Capped (winsorized) the outliers!')
print(f'   Max before: {max_before:.0f}')
print(f'   Max after : {max_after:.2f}')
print()
print('💡 Why cap, not delete?')
print('   We KEEP the customer (so we can predict their churn)')
print('   We PREVENT the extreme value from breaking statistics')

In [ ]:
# Visualize with box plots
print('📊 Box plots — best chart to see outliers')
print('   Box = middle 50%. Line in box = median. Dots = outliers.')
print()
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for i, col in enumerate(['monthly_charges', 'total_charges', 'tenure_months']):
    ax[i].boxplot(df[col].dropna())
    ax[i].set_title(col)
plt.tight_layout(); plt.show()
print()
print('💡 Lesson: BIG number ≠ WRONG number.')
print('   A 72-month customer is loyal, not an error.')

---
## Step 8: Save 💾

> **Rule:** Never overwrite the original file. Save with a NEW name.

In [ ]:
# Final check — everything OK?
print('🔍 Final check before saving:')
print()
print(f'   Shape: {df.shape}  (rows, columns)')
print(f'   Empty values: {df.isnull().sum().sum()}  ← should be 0!')
print()
print('   Column types:')
print(df.dtypes.value_counts())
print()
print('✅ Ready to save!')

In [ ]:
# Save to a NEW file
df.to_csv('customers_cleaned.csv', index=False)

print('✅ SAVED: customers_cleaned.csv')
print()
print('💡 We use index=False to NOT save row numbers as a column.')
print('   Otherwise we get an extra column with no name — ugly!')

---
## 🎁 Bonus: Apply to real data

We just cleaned synthetic data. **Same techniques work on real public datasets.**

We'll use **Telco Customer Churn** (a real public dataset). It's mostly clean but has 2 real problems we know how to fix.

👉 **Output of this section is what we use in Class 2 to train a model!**

In [ ]:
# Load real Telco data from the internet
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
telco = pd.read_csv(url)

print(f'📥 Loaded {len(telco)} customers from real Telco dataset')
print(f'   Columns: {telco.shape[1]}')
print()
print('💡 Telco is mostly clean (it has been used in classes for years)')
print('   But it has 2 real-world problems we know how to fix:')
print('   1. TotalCharges is text (should be number)')
print('   2. Verbose values like "No internet service"')

In [ ]:
# Fix 1: TotalCharges as text
before_type = telco['TotalCharges'].dtype
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce').fillna(0)

# Fix 2: collapse verbose values
verbose_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
for c in verbose_cols:
    telco[c] = telco[c].replace('No internet service', 'No')
telco['MultipleLines'] = telco['MultipleLines'].replace('No phone service', 'No')

# Save for Class 2
telco.to_csv('telco_churn_cleaned.csv', index=False)

print('✅ FIXED Telco data!')
print()
print(f'   TotalCharges type: {before_type} → {telco["TotalCharges"].dtype}')
print(f'   OnlineSecurity:    {list(telco["OnlineSecurity"].unique())}')
print()
print('💾 Saved: telco_churn_cleaned.csv')
print('   👉 Class 2 will use this file to train the model!')

---
## 🏁 We did it!

### What we cleaned today

| Step | Found | Fixed |
| --- | --- | --- |
| 2. Look | text where number should be, empty values, weird numbers | (just looking) |
| 3. Money | `' '` blanks in `total_charges` | text → number, fill |
| 4. Empty | 180 missing values | filled with median |
| 5. Words | 4 spellings of Tashkent | one spelling each |
| 6. Copies | 6 duplicates from merge | dropped |
| 7. Weird | tenure -3, 999, age 200, charges 99999 | replaced + capped |
| 8. Save | — | `customers_cleaned.csv` |
| Bonus | 2 real Telco problems | `telco_churn_cleaned.csv` |

### 🎯 Where we go next

- **Class 2:** Encoding + scaling (prepare for model)
- **Class 3:** Feature engineering (build new columns)
- **Module 4:** Train the model. Predict who will leave!

### 🎓 Words to remember

- **NaN** — empty value
- **dtype** — type of a column
- **Median** — middle value (safer than mean)
- **Outlier** — value far from the others
- **Clip / winsorize** — cap to a max

### 📤 Submit

1. Save as `Module3_Class1_<YourName>.ipynb`
2. PR to your group repo at `module-3/class_1/submissions/<YourName>/`

🧹 *Good job. See you in Class 2!*